# Unified Training Pipeline

This notebook integrates data preprocessing and training of a ResNet for both binary and multiclass motor fault classification.

In [ ]:
# Logging
import comet_ml
import yaml
import datetime

with open("../cfg.yaml", "r") as f:
    config = yaml.safe_load(f)

api_key = config['API_KEY']
project_name = config['PROJECT_NAME']
workspace = config['WORKSPACE']

experiment = comet_ml.start(api_key=api_key, 
                        project_name=project_name, 
                        workspace=workspace)

current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(current_time)
experiment.set_name(f"Run: {current_time}")
experiment.log_code()

In [ ]:
import numpy as np
import torch
import random

from src.data_pipeline import (
    generate_synthetic_peak, process_file, insert_synthetic_peaks, create_dataloaders, create_dataset
)
from src.electrical_signature_frequencies import ANOMALY_FREQS
from src.models import ResNet, ResidualBlock
from src.train import train_model
from src.inference import inference_resnet_model
from src.evaluation import calculate_metrics

from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Set device 
device: str = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
experiment.log_parameter("device", device)
device = torch.device(device)
print("Using device:", device)

## 1. Load Engine Configuration and Set Parameters

In [ ]:
# Load training parameters from train.yml
with open("../training_configs/train.yml", "r") as f:
    train_params = yaml.safe_load(f)

# Processing Parameters
f_sampling = train_params["processing_parameters"]["f_sampling"]
shift = train_params["processing_parameters"]["shift"]
segment_length = train_params["processing_parameters"]["segment_length"]
cutoff_freq = train_params["processing_parameters"]["cutoff_freq"]
peak_type = train_params["processing_parameters"]["peak_type"]
peak_segment = train_params["processing_parameters"]["peak_segment"]
amplitude_range = tuple(train_params["processing_parameters"]["amplitude_range"])

# Training Parameters
num_epochs_binary = train_params["training_parameters"]["num_epochs_binary"]
num_epochs_multi = train_params["training_parameters"]["num_epochs_multi"]

# Data Parameters
batch_size = train_params["data_parameters"]["batch_size"]
file_label_map = train_params["data_parameters"]["file_label_map"]
fault_types_available = train_params["data_parameters"]["fault_types_available"]

# Load Engine Configuration
with open('../engine_configs/LIMAN.yml', 'r') as f:
    engine_config = yaml.safe_load(f)

experiment.log_parameters({
    "f_sampling": f_sampling,
    "shift": shift,
    "segment_length": segment_length,
    "peak_segment": peak_segment,
    "cutoff_freq": cutoff_freq,
    "file_label_map": file_label_map,
    "fault_types_available": fault_types_available,
    "peak_type": peak_type,
    "num_epochs_binary": num_epochs_binary,
    "num_epochs_multi": num_epochs_multi,
    "batch_size": batch_size
})

fault_freqs = {
    fault_type: ANOMALY_FREQS.get(fault_type, lambda ec: [])(engine_config)
    for fault_type in fault_types_available
}
experiment.log_parameters(fault_freqs)

### Balancing Classes

Assume a file labeled "normal" produces $N$ segments. If you inject synthetic faults into a fraction $p$ of them, then:
- The number of unmodified (normal) segments is $(1-p)N$.
- The number of synthetic fault segments is $pN$.

If you have $k$ distinct fault types and you select each fault type uniformly at random during injection, then each fault type will get about $\frac{pN}{k}$ segments on average.

To have balanced classes across the $k+1$ classes (one normal and $k$ faults), you’d want:
$$
(1-p)N \approx \frac{pN}{k}\,,
$$
which simplifies to:
$$
1-p = \frac{p}{k} \quad \Longrightarrow \quad p \left( 1+\frac{1}{k} \right) = 1 \quad \Longrightarrow \quad p = \frac{k}{k+1}\,.
$$

For example:
- In binary classification (only one fault type, $k=1$), $p = \frac{1}{2} = 0.5$.
- In multiclass classification with, say, 3 fault types ($k=3$), $p = \frac{3}{4} = 0.75$.  
  This means that 75% of the segments from a normal file would be synthetically modified, and—if the injection is uniform—the normal class (25% of the segments) and each fault class (approximately 25% each) would be balanced.


In [ ]:
# Saving spectrums before and after peak injection

sample_file = None
for fp, lbl in file_label_map.items():
    if lbl == 'normal':
        sample_file = fp
        break

if sample_file is not None:
    try:
        segs, labels, freqs = process_file(sample_file, engine_config, 'normal',
                                            f_sampling=f_sampling, cutoff_freq=cutoff_freq,
                                            segment_length=segment_length, peak_segment=peak_segment, step=shift,
                                            apply_window=False, db=True,
                                            synthetic_fault_fraction=0.0, fault_types=None, amplitude_range=amplitude_range)
    except NameError:
        pass

    if segs is not None and len(segs) > 0:
        sample_segment = segs[0]
        for fault_type in fault_types_available:
            injected_segment = insert_synthetic_peaks(
                segment=sample_segment,
                fft_freqs=freqs,
                fault_type=fault_type,
                engine_config=engine_config,
                peak_segment=peak_segment,
                peak_generator=generate_synthetic_peak,
                sigma=2.0,
                method='gaussian'
            )
            difference = injected_segment - sample_segment
            fig, axs = plt.subplots(1, 3, figsize=(18, 5))
            axs[0].plot(freqs, sample_segment, color='blue')
            axs[0].set_title("Before Processing")
            axs[0].set_xlabel("Frequency")
            axs[0].set_ylabel("Magnitude (dB)")
            axs[0].grid(True)
            
            axs[1].plot(freqs, injected_segment, color='orange')
            axs[1].set_title(f"After Processing ({fault_type})")
            axs[1].set_xlabel("Frequency")
            axs[1].grid(True)
            
            axs[2].plot(freqs, difference, color='green')
            axs[2].set_title("Difference (After - Before)")
            axs[2].set_xlabel("Frequency")
            axs[2].grid(True)
            
            fig.suptitle(f"Spectrum Comparison for {fault_type}")
            experiment.log_figure(figure=fig, figure_name=f"spectrum_comparison_{fault_type}.png")
            plt.close(fig)

## 2. Train the ResNet Binary Classifier

In [ ]:
dataset_binary = create_dataset(file_label_map, engine_config, mode="binary",
                                f_sampling=f_sampling, cutoff_freq=cutoff_freq,
                                segment_length=segment_length, step=shift,
                                apply_window=False, db=True,
                                fault_types_available=fault_types_available,
                                include_real_anomalies_in_training=False, amplitude_range=amplitude_range,
                                add_noise=True)

print("Binary dataset shapes:", dataset_binary["X_train"].shape, dataset_binary["y_train"].shape, dataset_binary["X_test"].shape, dataset_binary["y_test"].shape)

train_loader_binary, val_loader_binary, test_loader_binary = create_dataloaders(dataset_binary, batch_size=batch_size)

In [ ]:
# Initialize ResNet
model_bin = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes=2, block_kwargs={'use_se': True}).to(device)
print("Training ResNet for binary classification...")
model_bin = train_model(model_bin, train_loader_binary, val_loader=val_loader_binary, device=device, num_epochs=num_epochs_binary)

In [ ]:
true_labels_bin, predictions_bin = inference_resnet_model(model_bin, test_loader_binary, device=device)

In [ ]:
# Calculate metrics 
cm_bin, acc_bin, prec_bin, rec_bin, f1_bin = calculate_metrics(true_labels_bin, predictions_bin)

In [ ]:
binary_report = classification_report(true_labels_bin, predictions_bin, target_names=['normal', 'anomalous'], output_dict=True)
binary_report_prefixed = {f"binary_{key}": value for key, value in binary_report.items()}
experiment.log_metrics(binary_report_prefixed)

In [ ]:
fig_bin, ax_bin = plt.subplots(figsize=(5,4))
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues',
            xticklabels=['normal','anomalous'], yticklabels=['normal','anomalous'], ax=ax_bin)
ax_bin.set_xlabel('Predicted Label')
ax_bin.set_ylabel('True Label')
ax_bin.set_title('Binary Classification - Confusion Matrix')
experiment.log_figure(figure=fig_bin, figure_name="confusion_matrix_binary.png")
plt.close(fig_bin)

## 3. Train the ResNet Multiclass Classifier

In [ ]:
dataset_multi = create_dataset(file_label_map, engine_config, mode="multiclass",
                               f_sampling=f_sampling, cutoff_freq=cutoff_freq,
                               segment_length=segment_length, peak_segment=peak_segment, step=shift,
                               apply_window=False, db=True,
                               fault_types_available=fault_types_available,
                               include_real_anomalies_in_training=False, amplitude_range=amplitude_range,
                               add_noise=True)

print("Multiclass dataset shapes:", dataset_multi["X_train"].shape, dataset_multi["y_train"].shape, dataset_multi["X_test"].shape, dataset_multi["y_test"].shape)
train_loader_multi, val_loader_multi, test_loader_multi = create_dataloaders(dataset_multi, batch_size=batch_size)

In [ ]:
# Train ResNet for Multiclass Classification
num_classes_multi = len(dataset_multi['label_encoder'].classes_)
model_multi = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes=num_classes_multi, block_kwargs={'use_se': True}).to(device)
print("Training ResNet for multiclass classification...")
model_multi = train_model(model_multi, train_loader_multi, val_loader=val_loader_multi, device=device, num_epochs=num_epochs_multi)

In [ ]:
true_labels_multi, predictions_multi = inference_resnet_model(model_multi, test_loader_multi, device=device)

In [ ]:
# Calculate metrics 
cm_multi, acc_multi, prec_multi, rec_multi, f1_multi = calculate_metrics(true_labels_multi, predictions_multi)

In [ ]:
multiclass_report = classification_report(true_labels_multi, predictions_multi, target_names=dataset_multi['label_encoder'].classes_, output_dict=True)
multiclass_report_prefixed = {f"multiclass_{key}": value for key, value in multiclass_report.items()}
experiment.log_metrics(multiclass_report_prefixed)

In [ ]:
fig_multi, ax_multi = plt.subplots(figsize=(5,4))
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues',
            xticklabels=dataset_multi['label_encoder'].classes_,
            yticklabels=dataset_multi['label_encoder'].classes_, ax=ax_multi)
ax_multi.set_xlabel('Predicted Label')
ax_multi.set_ylabel('True Label')
ax_multi.set_title('Multiclass Classification - Confusion Matrix')
experiment.log_figure(figure=fig_multi, figure_name="confusion_matrix_multiclass.png")
plt.close(fig_multi)

In [ ]:
experiment.end()